# 03 · Analysis (pre-declared)
- Dataset-level metrics: mean ± SD over 3 seeds.
- Paired per-image comparison of each connectivity loss vs. the baseline, per backbone (per-image metrics averaged over seeds): mean difference, bootstrap 95% CI, Wilcoxon signed-rank, Holm correction within each metric.
- **Consistency criterion (pre-declared):** a loss "consistently improves" a metric if the improvement holds (CI excludes 0 and Holm-adjusted p < 0.05) in **every** backbone. Betti-0 error is lower-is-better.
- Report null and negative results as they are.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
RESULTS = f'{PROJECT}/results'; OUT = f'{PROJECT}/analysis'
import sys, os; sys.path.insert(0, f'{PROJECT}/src'); os.makedirs(OUT, exist_ok=True)
from gwfss_stem import analysis
import pandas as pd
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 250)

In [ ]:
for tag in ['ss', 'ms']:
    runs = analysis.collect(RESULTS, tag=tag)
    if runs.empty: continue
    runs.to_csv(f'{OUT}/runs_{tag}.csv', index=False)
    tab = analysis.summary_table(runs); tab.to_csv(f'{OUT}/summary_{tag}.csv', index=False)
    print(f'=== {tag} ==='); display(tab)

In [ ]:
runs = analysis.collect(RESULTS, tag='ss')
tests = analysis.paired_tests(runs)
tests.to_csv(f'{OUT}/paired_tests_ss.csv', index=False)
cons = analysis.consistency(tests)
cons.to_csv(f'{OUT}/consistency_ss.csv', index=False)
display(tests.round(4)); display(cons)

## SQ3: resolution — ×2 training scale, and inference at ×1 (+flip), ×1.5, ×2 and multi-scale (×1, ×1.5, ×2 + flip). The per-scale decomposition (f10, s15, s20) is post hoc.

In [ ]:
INF = {'ss': 'x1', 'f10': 'x1 + flip', 's15': 'x1.5', 's20': 'x2', 'ms': 'x1,1.5,2 + flip'}
frames = []
for tag, label in INF.items():
    r = analysis.collect(RESULTS, tag)
    if r.empty: continue
    r = r[(r.split == 'random') & r.backbone.isin(['segformer_b1', 'deeplabv3plus_r101'])]
    frames.append(r.assign(inference=label))
sq3 = pd.concat(frames)
cols = ['iou_stem', 'boundary2px_iou_stem', 'cldice_stem', 'skel_recall_stem', 'betti0_err_stem_mean', 'miou']
display(sq3.groupby(['backbone', 'scale', 'loss', 'inference'])[[c for c in cols if c in sq3]].agg(['mean', 'std']).round(4))
sq3.to_csv(f'{OUT}/sq3_resolution.csv', index=False)

## Figures

In [ ]:
import matplotlib.pyplot as plt, numpy as np
s = analysis.collect(RESULTS, 'ss')
for split in s.split.unique():
    d = s[(s.split == split) & (s.scale == 1.0)]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, m in zip(axes, ['iou_stem', 'cldice_stem', 'betti0_err_stem_mean']):
        g = d.groupby(['backbone', 'loss'])[m].agg(['mean', 'std']).unstack('loss')
        g['mean'].plot.bar(yerr=g['std'], ax=ax, capsize=3, rot=20); ax.set_title(f'{split}: {m}')
    plt.tight_layout(); plt.savefig(f'{OUT}/bars_{split}.png', dpi=200); plt.show()

In [ ]:
# Qualitative comparison (seed 0 predictions): image | GT | base | clDice | Skeleton Recall
# Panels for runs that are not finished yet are left blank.
import cv2, glob
DATA_ROOT = '/content/gwfss'
from gwfss_stem import data; data.prepare_dataset(DATA_ROOT)
PALETTE = np.array([[0,0,0],[50,255,132],[50,132,255],[214,255,50]], np.uint8)
def colour(m): return PALETTE[np.where(m == 255, 0, m)]
BACKBONE, SPLIT = 'segformer_b1', 'random'
LOSSES = {'base': 'base', 'cldice': 'clDice', 'skelrecall': 'Skeleton Recall'}
base_dir = f'{RESULTS}/{SPLIT}/{BACKBONE}/base/scale1.0/seed0/pred_ss'
names = sorted(os.listdir(base_dir))[:6] if os.path.exists(base_dir) else []
for n in names:
    img = cv2.cvtColor(cv2.imread(f'{DATA_ROOT}/images/{n}'), cv2.COLOR_BGR2RGB)
    panels, titles = [img, colour(cv2.imread(f'{DATA_ROOT}/masks/{n}', cv2.IMREAD_UNCHANGED))], ['image', 'GT']
    for loss, label in LOSSES.items():
        pred = cv2.imread(f'{RESULTS}/{SPLIT}/{BACKBONE}/{loss}/scale1.0/seed0/pred_ss/{n}', cv2.IMREAD_UNCHANGED)
        panels.append(colour(pred) if pred is not None else np.full_like(img, 255))
        titles.append(label if pred is not None else f'{label} (not yet available)')
    fig, ax = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    for a, p, t in zip(ax, panels, titles):
        a.imshow(p); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT}/qual_{n}', dpi=150); plt.show()